In [30]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error,  mean_squared_error

In [2]:
DATA_PATH = "../../dataset/processed/feature_engineered.csv"
SEQ_LENGTH = 24
FORECAST_HORIZON = 6
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
BATCH_SIZE = 64
EPOCH = 100

In [3]:
df = pd.read_csv(DATA_PATH)

In [4]:
print("Dataset Shape:",df.shape)

Dataset Shape: (33476, 65)


In [5]:
df = df.select_dtypes(include=[np.number])

In [6]:
TARGET_COLUMNS = [
    "GA0151_A",
    "GA0151_C",
    "GA0151_D"
]

In [7]:
feature_columns = df.columns.tolist()
print("Number of features:", len(feature_columns))

Number of features: 65


In [8]:
n = len(df)
train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Total samples :", len(df))
print("Training      :", len(train_df))
print("Validation    :", len(val_df))
print("Testing       :", len(test_df))

Total samples : 33476
Training      : 23433
Validation    : 5021
Testing       : 5022


In [9]:
scaler = MinMaxScaler()
scaler.fit(train_df[feature_columns])
train_scaled = scaler.transform(train_df[feature_columns])
val_scaled = scaler.transform(val_df[feature_columns])
test_scaled = scaler.transform(test_df[feature_columns])

In [10]:
joblib.dump(scaler,"traffic_scaler.pkl")

['traffic_scaler.pkl']

In [11]:
def create_sequences(data, target_indices, seq_length, forecast_horizon):
    X = []
    y = []

    for i in range(seq_length, len(data) - forecast_horizon + 1):

        # Past 24 hours
        X.append(data[i - seq_length:i])

        # Next 6 hours for target sensors
        y.append(
            data[
                i:i + forecast_horizon,
                target_indices
            ]
        )

    return np.array(X), np.array(y)

In [12]:
target_indices = [
    feature_columns.index(col)
    for col in TARGET_COLUMNS
]

In [14]:
# Create training sequences
X_train, y_train = create_sequences(
    train_scaled,
    target_indices,
    SEQ_LENGTH,
    FORECAST_HORIZON
)

# Create validation sequences
X_val, y_val = create_sequences(
    val_scaled,
    target_indices,
    SEQ_LENGTH,
    FORECAST_HORIZON
)

# Create testing sequences
X_test, y_test = create_sequences(
    test_scaled,
    target_indices,
    SEQ_LENGTH,
    FORECAST_HORIZON
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (23404, 24, 65)
y_train: (23404, 6, 3)
X_val  : (4992, 24, 65)
y_val  : (4992, 6, 3)
X_test : (4993, 24, 65)
y_test : (4993, 6, 3)


In [18]:
model = Sequential([
    Input(shape=(SEQ_LENGTH, len(feature_columns))),
    LSTM(128, return_sequences=True),
    Dropout(0.2),

    LSTM(64, return_sequences=False),
    Dropout(0.2),

    Dense(64, activation="relu"),

    Dense(FORECAST_HORIZON * len(TARGET_COLUMNS))
])

In [19]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 24, 128)        │        99,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 18)             │         1,170 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 154,066 (601.82 KB)

 Trainable params: 154,066 (601.82 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
y_train = y_train.reshape(
    y_train.shape[0],
    FORECAST_HORIZON * len(TARGET_COLUMNS)
)

y_val = y_val.reshape(
    y_val.shape[0],
    FORECAST_HORIZON * len(TARGET_COLUMNS)
)

y_test = y_test.reshape(
    y_test.shape[0],
    FORECAST_HORIZON * len(TARGET_COLUMNS)
)


print("\nAfter reshaping:")
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)


After reshaping:
y_train: (23404, 18)
y_val  : (4992, 18)
y_test : (4993, 18)


In [21]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    mode="min"
)

In [22]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(
        X_val,
        y_val
    ),
    epochs=EPOCH,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping,
        checkpoint
    ],
    verbose=1
)

Epoch 1/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 0.0093 - mae: 0.0664 - val_loss: 0.0094 - val_mae: 0.0656
Epoch 2/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.0058 - mae: 0.0517 - val_loss: 0.0084 - val_mae: 0.0609
Epoch 3/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.0050 - mae: 0.0474 - val_loss: 0.0074 - val_mae: 0.0563
Epoch 4/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.0046 - mae: 0.0453 - val_loss: 0.0070 - val_mae: 0.0573
Epoch 5/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 14s 37ms/step - loss: 0.0044 - mae: 0.0440 - val_loss: 0.0073 - val_mae: 0.0589
Epoch 6/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - loss: 0.0042 - mae: 0.0429 - val_loss: 0.0067 - val_mae: 0.0543
Epoch 7/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.0041 - mae: 0.0421 - val_loss: 0.0069 - val_mae: 0.0557
Epoch 8/100
366/366 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - loss: 0.0039 - mae: 0.0413 - val_loss: 0.0069 - val_mae: 0.0558
Epoch 9/100
366/366 ━━━━━━━━━━━━━━━━━

In [23]:
test_loss, test_mae = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("\nTest Loss (MSE):", test_loss)
print("Test MAE:", test_mae)


157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0081 - mae: 0.0616

Test Loss (MSE): 0.008102054707705975
Test MAE: 0.06155913323163986


In [24]:
y_pred = model.predict(X_test)
print("\nPrediction shape:")
print("y_pred:", y_pred.shape)

157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step

Prediction shape:
y_pred: (4993, 18)


In [25]:
y_pred = y_pred.reshape(
    -1,
    FORECAST_HORIZON,
    len(TARGET_COLUMNS)
)

y_test_original = y_test.reshape(
    -1,
    FORECAST_HORIZON,
    len(TARGET_COLUMNS)
)


print("\nFinal shapes:")
print("y_pred:", y_pred.shape)
print("y_test:", y_test_original.shape)


Final shapes:
y_pred: (4993, 6, 3)
y_test: (4993, 6, 3)


In [26]:
num_features = len(feature_columns)

y_pred_original = np.zeros(
    (
        y_pred.shape[0],
        FORECAST_HORIZON,
        num_features
    )
)

y_test_original_scaled = np.zeros(
    (
        y_test_original.shape[0],
        FORECAST_HORIZON,
        num_features
    )
)

In [28]:
for j, target_idx in enumerate(target_indices):

    y_pred_original[:, :, target_idx] = y_pred[:, :, j]

    y_test_original_scaled[:, :, target_idx] = (
        y_test_original[:, :, j]
    )

# Inverse scaling
y_pred_original = scaler.inverse_transform(
    y_pred_original.reshape(-1, num_features)
).reshape(
    -1,
    FORECAST_HORIZON,
    num_features
)

y_test_original = scaler.inverse_transform(
    y_test_original_scaled.reshape(-1, num_features)
).reshape(
    -1,
    FORECAST_HORIZON,
    num_features
)


In [29]:
y_pred_original = y_pred_original[:, :, target_indices]

y_test_original = y_test_original[:, :, target_indices]


print("\nOriginal-scale shapes:")
print("Predictions:", y_pred_original.shape)
print("Actual     :", y_test_original.shape)



Original-scale shapes:
Predictions: (4993, 6, 3)
Actual     : (4993, 6, 3)


In [31]:
for i, sensor in enumerate(TARGET_COLUMNS):

    actual = y_test_original[:, :, i].flatten()
    predicted = y_pred_original[:, :, i].flatten()

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    r2 = r2_score(
        actual,
        predicted
    )

    # Avoid division by zero for MAPE
    mask = actual != 0

    mape = np.mean(
        np.abs(
            (actual[mask] - predicted[mask])
            / actual[mask]
        )
    ) * 100

    print("\n--------------------------------")
    print("Sensor:", sensor)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("MAPE:", mape, "%")
    print("R²  :", r2)


--------------------------------
Sensor: GA0151_A
MAE : 9.429544505020386
RMSE: 15.588649669879732
MAPE: 59.55204974947138 %
R²  : 0.6376061338396064

--------------------------------
Sensor: GA0151_C
MAE : 24.524494744482265
RMSE: 33.7700285285136
MAPE: 21.3071458244708 %
R²  : 0.8006430634561844

--------------------------------
Sensor: GA0151_D
MAE : 22.425943258000604
RMSE: 30.573875075660613
MAPE: 30.72290564397014 %
R²  : 0.7378965599546266
